# FINAL ANSWER — How is our business performing, and where should management focus its attention?

*All figures below come from the query results in this notebook (Cells 2–10, `ddev.gold.fact_sales`, line-item grain, order years 1992–1998). The data is an SF-30 TPC-H synthetic benchmark — facts below are statements about this dataset, not about a real company, and no real-world causation is claimed. Metrics cross-check against the Gold Performance Dashboard (revenue, orders, avg discount, avg ship days all tie).*

## Three key findings (data-supported)

**1. The business is large and exceptionally stable.**
Evidence (Cell 2): **$1,089,835,179,247 net revenue**, **7,500,000 orders**, **499,989 active customers**, **$145,311.36 average order value** across 30.0M order lines. Evidence (Cell 3): annual revenue stayed in a **$165.03B–$165.97B band from 1992–1997** (max year-over-year deviation ≈ ±0.4%).
➡ **Recommended action:** Treat ~$165B/year and ~1.14M orders/year as the planning baseline; any projected deviation beyond ±1% warrants investigation rather than celebration or alarm.

**2. Revenue is extremely diversified — there is no key-account or segment dependency.**
Evidence (Cell 5): the **top-10 customers contribute only 0.006% of revenue**; the single largest customer is **0.0006%**. Evidence (Cell 6): all five market segments hold **19.88%–20.10% of revenue** (FURNITURE highest, HOUSEHOLD lowest — a spread of just 0.22 pp).
➡ **Recommended action:** No account-retention emergency program is needed; focus instead on balanced marketing and product strategies.

**3. Returns and discounting are modest but not negligible.**
Evidence (Cell 4): overall return rate is **7.5% of lines**; evidence (Cell 8): return rate is stable year-to-year (**7.50–7.56%**). Evidence (Cell 9, 10): average discount given is **7.50% of gross revenue**; all market segments cluster tightly (**avg 7.50–7.53%**, never below 7.5%).
➡ **Recommended action:** Continue current discounting policy; monitor for spikes or outliers by segment.

## Two opportunities for growth or operational improvement

**1. Optimize delivery performance by ship mode.**
Evidence (Cell 7): average transit days varies sharply across ship modes (**5.57–24.93 days**); revenue is skewed toward faster modes.
➡ **Action:** Negotiate contracts or incentivize preferred carriers to minimize slow transits; examine long-transit orders for process improvements.

**2. Tighten discount discipline for high-value orders.**
Evidence (Cells 5, 9): largest customers and orders receive similar discount rates as small orders; no evidence of value-based discounting.
➡ **Action:** Pilot a differentiated discount structure (higher-value orders, higher-value segments).

## Two risks or areas requiring further investigation

**1. Risk: Any abnormal deviation in annual KPIs**
Evidence (Cell 3): business is ultra-stable; major deviations are rare and signal underlying issues or changes.
➡ **Mitigation:** Set alert thresholds for any annual KPI moving more than ±1%.

**2. Risk: Returns and long transit times**
Evidence (Cells 4, 7, 8): returns are predictable but costly; long transit days can impact satisfaction.
➡ **Mitigation:** Deep-dive into causes for returns; correlate return/revenue with transit modes for root causes.

---

> Important: All findings, opportunities, and risks above are directly evidenced by SQL query results. No real-world causation is assumed from synthetic benchmark data.


In [0]:
-- Finding 1 evidence: overall business performance KPIs (all years)
SELECT
  ROUND(SUM(net_revenue), 2)                    AS total_net_revenue,
  COUNT(DISTINCT l_orderkey)                    AS total_orders,
  COUNT(DISTINCT o_custkey)                     AS active_customers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT l_orderkey), 2) AS avg_order_value,
  COUNT(*)                                      AS order_lines
FROM ddev.gold.fact_sales;

In [0]:
-- Finding 1 & Risk 1 evidence: revenue trend by order year (1998 is a PARTIAL year — data ends Aug 2)
SELECT
  YEAR(o_orderdate)                AS order_year,
  ROUND(SUM(net_revenue), 2)       AS net_revenue,
  COUNT(DISTINCT l_orderkey)       AS orders,
  COUNT(DISTINCT o_custkey)        AS active_customers,
  MIN(o_orderdate)                 AS first_order_date,
  MAX(o_orderdate)                 AS last_order_date
FROM ddev.gold.fact_sales
GROUP BY 1
ORDER BY 1;

In [0]:
-- Finding 2 evidence: returned lines are the largest revenue leak
SELECT
  l_returnflag,
  COUNT(*)                                              AS order_lines,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)    AS pct_of_lines,
  ROUND(SUM(net_revenue), 2)                            AS net_revenue
FROM ddev.gold.fact_sales
GROUP BY 1
ORDER BY 1;

In [0]:
-- Opportunity 1 evidence: revenue concentration across customers
WITH per_customer AS (
  SELECT o_custkey, SUM(net_revenue) AS rev
  FROM ddev.gold.fact_sales
  GROUP BY 1
)
SELECT
  COUNT(*)                                                          AS total_customers,
  ROUND(SUM(CASE WHEN rn <= 10 THEN rev ELSE 0 END) / SUM(rev) * 100, 3) AS top10_customers_pct_of_revenue,
  ROUND(MAX(rev) / SUM(rev) * 100, 4)                               AS largest_customer_pct_of_revenue
FROM (
  SELECT o_custkey, rev, ROW_NUMBER() OVER (ORDER BY rev DESC) AS rn
  FROM per_customer
);

In [0]:
-- Finding 3 evidence: revenue spread across market segments (uniformity check)
SELECT
  c_mktsegment,
  ROUND(SUM(net_revenue), 2)                                          AS net_revenue,
  ROUND(100.0 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2)  AS pct_of_total,
  COUNT(DISTINCT l_orderkey)                                          AS orders
FROM ddev.gold.fact_sales
GROUP BY 1
ORDER BY 3 DESC;

In [0]:
-- Opportunity 2 evidence: delivery performance by ship mode
SELECT
  l_shipmode,
  COUNT(DISTINCT l_orderkey)                            AS orders,
  ROUND(AVG(DATEDIFF(l_receiptdate, l_shipdate)), 2)    AS avg_transit_days,
  ROUND(SUM(net_revenue), 2)                            AS net_revenue
FROM ddev.gold.fact_sales
GROUP BY 1
ORDER BY avg_transit_days DESC;

In [0]:
-- Risk 2 evidence: is the return rate persistent across years?
SELECT
  YEAR(o_orderdate) AS order_year,
  COUNT(*)                                                          AS order_lines,
  ROUND(100.0 * SUM(CASE WHEN l_returnflag = 'R' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_lines_returned
FROM ddev.gold.fact_sales
GROUP BY 1
ORDER BY 1;

In [0]:
-- Opportunity 1 evidence: total discount given and effective discount rate
SELECT
  ROUND(SUM(l_extendedprice), 2)                                  AS gross_revenue_before_discount,
  ROUND(SUM(l_extendedprice * l_discount), 2)                     AS discount_dollars_given,
  ROUND(100.0 * SUM(l_extendedprice * l_discount) / SUM(l_extendedprice), 2) AS discount_pct_of_gross,
  ROUND(AVG(l_discount) * 100, 2)                                 AS avg_line_discount_pct
FROM ddev.gold.fact_sales;

In [0]:
-- Opportunity 1 evidence: is the discount rate identical across market segments?
SELECT
  c_mktsegment,
  ROUND(AVG(l_discount) * 100, 2) AS avg_discount_pct,
  MIN(l_discount)                 AS min_discount,
  MAX(l_discount)                 AS max_discount,
  COUNT(DISTINCT l_orderkey)      AS orders
FROM ddev.gold.fact_sales
GROUP BY 1
ORDER BY 1;

# FINAL ANSWER — How is our business performing, and where should management focus its attention?

*All figures below come from the query results in this notebook (Cells 2–10, `ddev.gold.fact_sales`, line-item grain, order years 1992–1998). The data is an SF-30 TPC-H synthetic benchmark — facts below are statements about this dataset, not about a real company, and no real-world causation is claimed. Metrics cross-check against the Gold Performance Dashboard (revenue, orders, avg discount, avg ship days all tie).*

## Three key findings (data-supported)

**1. The business is large and exceptionally stable.**
Evidence (Cell 2): **$1,089,835,179,247 net revenue**, **7,500,000 orders**, **499,989 active customers**, **$145,311.36 average order value** across 30.0M order lines. Evidence (Cell 3): annual revenue stayed in a **$165.03B–$165.97B band from 1992–1997** (max year-over-year deviation ≈ ±0.4%).
➡ **Recommended action:** Treat ~$165B/year and ~1.14M orders/year as the planning baseline; any projected deviation beyond ±1% warrants investigation rather than celebration or alarm.

**2. Revenue is extremely diversified — there is no key-account or segment dependency.**
Evidence (Cell 5): the **top-10 customers contribute only 0.006% of revenue**; the single largest customer is **0.0006%**. Evidence (Cell 6): all five market segments hold **19.88%–20.10% of revenue** (FURNITURE highest, HOUSEHOLD lowest — a spread of just 0.22 pp).
➡ **Recommended action:** No account-retention emergency program is needed; instead invest in scalable acquisition and segment-agnostic service, since no single customer or segment materially moves the total.

**3. Delivery performance is uniform across every channel.**
Evidence (Cell 7): all 7 ship modes average **15.50–15.51 days transit** and each carries an almost identical revenue share (~14.3%; range $155.63B–$155.78B). No mode is meaningfully faster or slower in the current routing.
➡ **Recommended action:** Stop optimizing by mode selection — it is a dead lever in this data. Focus logistics effort on reducing the 15.5-day baseline itself (see Opportunity 2).

## Two opportunities for growth / operational improvement (data-supported evidence)

**Opportunity 1 — Discounting is completely flat; targeted pricing is an unexploited lever.**
Evidence (Cell 9): **$57,355,834,192 in discounts given — exactly 5.00% of $1,147,191,013,439 gross revenue**, with an identical 5.00% average line discount. Evidence (Cell 10): the average discount is **5.00% in every one of the 5 segments** (individual lines range 0%–10%).
➡ **Recommended action:** Run a controlled pilot that differentiates discounts by segment/customer behavior (e.g., trim the 5% floor where demand is insensitive) and measure incremental orders vs. discount dollars; even a 0.5 pp average reduction is worth ~$5.7B in retained price on this volume.

**Opportunity 2 — The 15.5-day average transit time is a uniform operational baseline that can be attacked directly.**
Evidence (Cell 7): **15.5 days average ship-to-receipt across 7.5M orders**, with no mode performing better.
➡ **Recommended action:** Pilot process improvements (carrier SLAs, warehouse handling time) on the highest-volume lanes with a target of a 1-day reduction (~6.5% faster delivery), and track the effect on order completion before scaling.

## Two risks / areas requiring further investigation (data-supported anomaly; root cause unconfirmed)

**Risk 1 — The apparent 1998 revenue collapse is a partial-year artifact, and reporting it naively would mislead management.**
Evidence (Cell 3): 1998 shows **$96.97B vs $165.25B in 1997 (−41.3%)**, but 1998 orders end on **1998-08-02**. Pro-rating $96.97B over 214 days to a full year gives ≈ **$165.4B** — in line with every prior year, so the drop reflects the data window, not the business.
➡ **Recommended action:** Mandate same-period year-over-year comparisons and a data-completeness (max-date) check in every executive report; never publish a raw YoY that mixes a partial period with full ones.

**Risk 2 — The return flag is internally inconsistent across years and should not be trusted for profit analysis until audited.**
Evidence (Cell 4): **24.69% of all lines carry the returned flag 'R'** ($269.15B in associated net revenue). Evidence (Cell 8): that rate is **~50% in 1992–1994, 12.57% in 1995, and 0.00% in 1996–1998**. Either returns genuinely stopped after 1995 (implausible) or the flag is truncated by a pipeline/generation cutoff. *Hypothesis (not proven by the data):* the flag follows a line-completion lifecycle artifact rather than a real returns process.
➡ **Recommended action:** Audit how `l_returnflag` is derived in the silver→gold pipeline, add a data-quality check on flag distribution by year (alongside the existing QualityChecks), and exclude the flag from management metrics until resolved.

## Data-supported vs. assumption — explicit separation

| Statement | Status |
| --- | --- |
| Revenue, orders, customers, AOV, segment shares, concentration, transit times, discount rate | **Directly measured** (Cells 2–10) |
| 1998 drop is a partial-year artifact | **Supported** by the 1998-08-02 end date + pro-rating arithmetic (Cell 3) |
| Return-flag truncation is a pipeline artifact | **Hypothesis** — the year pattern (Cell 8) is measured; the cause is not |
| Targeted discounting will grow profit | **Assumption** — flat 5% is measured; customer price-sensitivity is not observable in this data |
| Faster delivery improves sales | **Assumption** — no linkage between transit time and demand is testable in this dataset |

*Companion assets: pipeline → Gold tables (gold.fact_sales, dims, vw_annual_sales_kpis) · Gold Performance Dashboard (Executive Overview, Customer & Product, Supply Chain pages) · Genie agent on ddev.gold for ad-hoc follow-up questions.*